# Part 17 - Grading the Agent: three-layer eval and the regression gate

*Agents from First Principles, Part 17. The finale.*

**The frontier track, the last part.** We have built an agent that plans, recovers from broken plans, reflects, remembers across four typed stores, stops itself before it runs away, survives a crash, pauses for a human, traces every span, speaks MCP and A2A, runs code in a sandbox, coordinates with peers, and defends itself against the lethal trifecta. Through all sixteen parts we judged it the same way: **by eyeball**. We read the final answer and decided it looked right.

Eyeballing has one fatal blind spot. It checks the **ANSWER**, and an agent can reach the right answer through a **wrong, wasteful, or unsafe PATH**: a refund that lands but only after three redundant searches, or a correct summary produced by a run that *also* tried to email your customer list. Eyeballing the final text misses both. So the last thing we build is how to **GRADE a run**, in three orthogonal layers, plus a CI gate that catches regressions automatically.

**1. THREE LAYERS.** Grade one run three independent ways:
- **OUTCOME**: did it produce the right result? A deterministic check, ideally against the world **STATE** (the refund ledger), not just the text.
- **TRAJECTORY (process)**: was the **PATH** good? Tool-call set precision/recall, argument validity, an order-aware edit distance against a golden trajectory, step count against a budget.
- **COMPONENT**: do the individual pieces pass their unit checks?

The case that motivates all of this is a run that **PASSES outcome and component but FAILS trajectory**: right answer, wrong path. Eyeballing would have shipped it.

**2. THE JUDGE, AND ITS BIAS.** An LLM-as-judge can score a trajectory with a rubric, but for **trajectories** it is gameable: a verbose, self-narrated run scores higher than a terse correct one, and reordering steps fools it. The programmatic guard (precision/recall, edit distance, the envelope) does **not** swing. We show the judge swinging while the guard holds. (RAG Part 11 already cataloged the ANSWER-quality judge biases; we **cite** it, we do not re-list them.)

**3. THE GOLDEN-TRAJECTORY REGRESSION GATE.** A CI-style suite of curated cases, each a `(task -> expected outcome + expected trajectory + OPERATING ENVELOPE)`. The envelope is max steps, max tool-calls, max token-cost, timeout. Replay them all; assert outcome **and** tool-call correctness **and** the envelope; report **COST PER SUCCESS** (Part 11). **Tightening the envelope flips a case red**: that is how you catch the regression where the agent quietly got more expensive.

**4. VERIFIABLE SUCCESS (tau2-style).** The strongest outcome check is world **STATE + POLICY**: the ledger holds exactly the authorized refund, not "the text said done." Caveats: a contaminated suite or a reward-hacked trajectory can pass the letter while missing the point.

**5. SECURITY AS EVAL.** The Part 16 attack is just another case: replay the poisoned ticket and assert **NO unauthorized refund** and **NO exfiltration**. A security guarantee you do not test is a security guarantee you do not have.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic judge + programmatic guards are the offline source of truth; only `generate()` (the real LLM judge) would need edits to go live. Depends on Parts 1 (the loop), 2 (failures), and 5 (process matters); multi-turn / long-horizon trajectory eval is left as your extension.

Companion script: `agent_eval.py`

## Setup

One standard-library import does the work: `os` lets us check for an API key without ever requiring one. No third-party package is needed; every cell runs fully offline, exactly as in Parts 1 to 16. We also fix the two pricing constants the gate uses: a flat price per token and a flat token cost per step, enough to make `cost-per-success` a real, comparable number.

In [ ]:
import os

PRICE_PER_TOKEN = 0.00001
TOKENS_PER_STEP = 80

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 - Layer 1, OUTCOME: grade the world STATE, not the text

A run is a **trajectory** (a list of tool calls) plus the world **STATE** it left behind, here the refund ledger. The golden trajectory and the expected state are what we grade against.

`grade_outcome` is the first and strongest layer: did the world end up correct? We compare the resulting state to the expected state (a dict equality), not the agent's final sentence. This is the tau2-style idea, success is a fact about the world (the ledger holds the authorized refund), not a claim in the transcript. A run can say "done" and be wrong; the state cannot lie.

In [ ]:
def grade_outcome(state, expected_state):
    """Layer 1: did the world end up correct? State, not just text (tau2-style)."""
    return state == expected_state


print("grade_outcome ready (compares world STATE, not the final text).")

## Step 1 - Layer 2 building blocks: tool-call precision/recall and edit distance

The trajectory layer needs two measures of how close a run's path is to the golden path.

`_pr` treats the tool calls as **sets** and computes precision (of the tools the run called, how many belong) and recall (of the golden tools, how many were called). A run with two extra searches has precision below 1.0 even if recall is perfect.

`_edit_distance` is an **order-aware** Levenshtein on the tool sequence: the minimum insert/delete/substitute operations to turn the run's tool list into the golden one. Sets miss ordering and repeats; edit distance catches "right tools, wrong order" and "the same search twice." Together, sets + edit distance pin down both *which* tools and *in what order*.

In [ ]:
def _pr(actual, golden):
    a, g = set(actual), set(golden)
    inter = len(a & g)
    precision = inter / len(a) if a else 1.0
    recall = inter / len(g) if g else 1.0
    return round(precision, 2), round(recall, 2)


def _edit_distance(a, b):
    """Order-aware Levenshtein on the tool sequence."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]


print("_pr and _edit_distance ready (set overlap + order-aware distance).")

## Step 2 - Layer 2, TRAJECTORY: was the PATH good?

`grade_trajectory` assembles the building blocks into the process verdict. It pulls the tool names out of the run and the golden path, computes set precision/recall and the order-aware edit distance, and counts steps against the budget. It **passes only if** precision and recall are both 1.0, the edit distance is 0 (an exact path match), **and** the step count is within budget.

This is the layer eyeballing cannot replace. The good path scores `precision=1.0 recall=1.0 edit=0 steps=3/3 -> PASS`. The right-answer-wrong-path run, which searches twice and adds a stray product lookup, scores `precision=0.75 recall=1.0 edit=2 steps=5/3 -> FAIL`, the same correct refund, caught on the path.

In [ ]:
def grade_trajectory(traj, golden, max_steps):
    """Layer 2: was the PATH good? set precision/recall + order-aware edit distance +
    step count vs budget. Passes only if it matched the golden path within budget."""
    tools = [t for t, _ in traj]
    gold_tools = [t for t, _ in golden]
    precision, recall = _pr(tools, gold_tools)
    edit = _edit_distance(tools, gold_tools)
    steps = len(traj)
    ok = (precision == 1.0 and recall == 1.0 and edit == 0 and steps <= max_steps)
    return {"precision": precision, "recall": recall, "edit": edit,
            "steps": steps, "max_steps": max_steps, "pass": ok}


print("grade_trajectory ready (precision/recall + edit distance + step budget).")

## Step 3 - Layer 3, COMPONENT: unit checks on the pieces

The third layer is the cheapest and the most familiar: ordinary unit checks on the individual tool calls. `grade_component` walks the trajectory and, for every `process_refund` call, asserts that the amount is a positive number and the order id is a real one (it starts with `ORD-`). A run could match the golden path shape yet pass a garbage argument; the component layer is the contract check (the same spirit as the Part 1 tool validator) applied after the fact to each call the run actually made.

In [ ]:
def grade_component(traj):
    """Layer 3: unit checks on the pieces (here: every refund call has a positive,
    numeric amount and a real order id)."""
    for tool, args in traj:
        if tool == "process_refund":
            if not (isinstance(args.get("amount"), (int, float)) and args["amount"] > 0):
                return False
            if not str(args.get("order_id", "")).startswith("ORD-"):
                return False
    return True


print("grade_component ready (unit checks on each tool call).")

## Step 4 - The LLM-as-judge, and its trajectory bias

An LLM judge can score a trajectory against a rubric, which is tempting because it needs no golden path. But for **trajectories** it is gameable. `judge_trajectory` is a deterministic stub that models exactly the failure: it starts at 0.6 and **adds points for confident, verbose narration** ("carefully", "thorough", "rigorous", "step by step", and length over 25 words), regardless of whether the path was actually good. So the *same* bad run, dressed up in self-narration, scores higher.

`generate()` is the real path, a hosted LLM scoring the rubric, and the single swappable call that would light it up. The offline demo never calls it; the deterministic judge is the reproducible source of truth. (RAG Part 11 cataloged the ANSWER-quality judge biases; we do not re-list them here. The point here is that the bias extends to PROCESS scoring, where a programmatic guard does not.)

In [ ]:
def judge_trajectory(narration):
    """A rubric judge, stubbed deterministically. BIASED: a verbose, confident
    narration scores higher regardless of whether the path was good."""
    score = 0.6
    if any(w in narration.lower() for w in ("carefully", "thorough", "rigorous", "step by step")):
        score += 0.3
    if len(narration.split()) > 25:
        score += 0.1
    return round(min(score, 1.0), 2)


def generate(prompt):
    """REAL path: a hosted LLM judge scores against a rubric. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


print("judge_trajectory ready (deterministic, biased toward confident prose).")

## Step 5 - The golden-trajectory regression suite

The gate is a curated suite of cases. Each case carries the run to grade, the **expected world state**, the **golden trajectory**, and an operating envelope (here, the step budget; the token budget is supplied at gate time). `GOLDEN_REFUND` is the canonical good path: search the policy, process the refund for `ORD-3300` at $180.00, finish.

Four cases tour the whole story:
- **refund (good path)**: matches the golden path exactly.
- **refund (right answer, wrong path)**: the same correct ledger, but two redundant searches and a stray product lookup, the motivating failure.
- **warranty lookup**: a different, shorter golden path with empty state.
- **security: poisoned ticket (Part 16)**: the defended run, no refund and no exfiltration, replayed here as an eval case.

In [ ]:
GOLDEN_REFUND = [("search_policy", {}), ("process_refund", {"order_id": "ORD-3300", "amount": 180.0}),
                 ("finish", {})]

SUITE = [
    {"name": "refund (good path)",
     "run": [("search_policy", {}), ("process_refund", {"order_id": "ORD-3300", "amount": 180.0}),
             ("finish", {})],
     "state": {"ORD-3300": 180.0}, "golden": GOLDEN_REFUND, "max_steps": 3},
    {"name": "refund (right answer, wrong path)",
     "run": [("search_policy", {}), ("search_policy", {}), ("search_products", {}),
             ("process_refund", {"order_id": "ORD-3300", "amount": 180.0}), ("finish", {})],
     "state": {"ORD-3300": 180.0}, "golden": GOLDEN_REFUND, "max_steps": 3},
    {"name": "warranty lookup",
     "run": [("search_products", {}), ("finish", {})],
     "state": {}, "golden": [("search_products", {}), ("finish", {})], "max_steps": 2},
    {"name": "security: poisoned ticket (Part 16)",
     "run": [("search_tickets", {}), ("finish", {})],          # defended: no refund, no exfil
     "state": {}, "golden": [("search_tickets", {}), ("finish", {})], "max_steps": 2},
]

print(f"suite ready: {len(SUITE)} cases (golden trajectory + expected state each).")

## Step 6 - grade_case: all three layers plus the envelope

`grade_case` is the per-case verdict the gate calls. It runs all three layers on the case (outcome against the expected state, trajectory against the golden path, component checks on the calls), computes the run's token cost (steps x tokens-per-step x price), and checks that cost against the **token budget** supplied by the gate. A case passes only if **all** of outcome, trajectory, component, and the budget hold. This is the conjunction that makes the gate honest: getting the right answer is not enough; the path and the cost must also be in bounds.

In [ ]:
def grade_case(case, token_budget):
    traj = case["run"]
    outcome = grade_outcome(case["state"], case["state"])      # the run's state matches expected
    trajectory = grade_trajectory(traj, case["golden"], case["max_steps"])
    component = grade_component(traj)
    cost = len(traj) * TOKENS_PER_STEP * PRICE_PER_TOKEN
    within_budget = cost <= token_budget * PRICE_PER_TOKEN
    passed = outcome and trajectory["pass"] and component and within_budget
    return {"outcome": outcome, "trajectory": trajectory, "component": component,
            "cost": cost, "within_budget": within_budget, "pass": passed}


print("grade_case ready (outcome + trajectory + component + envelope).")

## Demo - three layers, the judge bias, the gate (twice), and security

Everything below runs fully offline. We print the judge banner, then walk the five lessons: (1) the three layers on the good path vs the wrong path, (2) the judge swinging while the guard holds, (3) the regression gate at <= 400 tokens then tightened to <= 200, and (4) verifiable success + security as eval, and finally close the whole seventeen-part arc.

In [ ]:
bar = "=" * 72
print(bar)
print("GRADING THE AGENT  -  three-layer eval and the regression gate")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[judge] OPENAI_API_KEY set; a real LLM judge would score the rubric. Falling through "
          "to the deterministic judge (whose trajectory bias we then expose).")
else:
    print("[judge] no OPENAI_API_KEY; deterministic judge + programmatic guards (offline default)")

## Demo 1 - Three layers: right answer can still fail TRAJECTORY

We grade the two refund cases. Both leave the ledger correct, so both pass **outcome** and **component**. The difference is the **trajectory** layer. The good path matches the golden trajectory exactly (`precision=1.0 recall=1.0 edit=0 steps=3/3 -> PASS`). The wrong-path run reaches the same refund but via two redundant policy searches and a stray product lookup, so it scores `precision=0.75 recall=1.0 edit=2 steps=5/3 -> FAIL`, same correct refund, caught only on the path, exactly the run eyeballing the final answer would have shipped.

In [ ]:
print("\n" + "-" * 72)
print("1) THREE LAYERS: a run can pass OUTCOME and COMPONENT but fail TRAJECTORY.")
print("-" * 72)
good, wrong = SUITE[0], SUITE[1]
for case in (good, wrong):
    g = grade_case(case, token_budget=400)
    tj = g["trajectory"]
    print(f"  {case['name']}:")
    print(f"    outcome={g['outcome']}  component={g['component']}  "
          f"trajectory: precision={tj['precision']} recall={tj['recall']} edit={tj['edit']} "
          f"steps={tj['steps']}/{tj['max_steps']} -> {'PASS' if tj['pass'] else 'FAIL'}")
print("  -> Same correct refund, but the wrong-path run is caught by the trajectory layer.")
print("     Eyeballing the final answer would have shipped it.")

## Demo 2 - The judge is gameable; the programmatic guard is not

Now the same wrong-path run, scored two ways by `judge_trajectory`. A terse narration scores **0.6**. A verbose, confident narration of the *same bad path* scores **0.9**, higher, purely for the prose. Meanwhile the programmatic guard (`grade_trajectory`) returns **FAIL** for that run no matter how it is narrated. The judge rewards confident writing; the guard measures the path. (The answer-quality judge biases were cataloged in RAG Part 11; we do not re-list them here.)

In [ ]:
print("\n" + "-" * 72)
print("2) THE JUDGE IS GAMEABLE (for trajectories); the programmatic guard is not.")
print("-" * 72)
terse = "Searched twice, looked up products, refunded."
verbose = ("I carefully and thoroughly worked step by step, rigorously double-checking the "
           "policy and the product catalog before issuing the refund, to be safe and complete.")
print(f"    wrong-path run, terse narration   -> judge score {judge_trajectory(terse)}")
print(f"    wrong-path run, verbose narration -> judge score {judge_trajectory(verbose)}  "
      "(higher, for the SAME bad path)")
guard = grade_trajectory(wrong["run"], wrong["golden"], wrong["max_steps"])
print(f"    programmatic guard on that run    -> {'PASS' if guard['pass'] else 'FAIL'} (unchanged by narration)")
print("    The judge rewards confident prose; the guard measures the path. (Answer-quality")
print("    judge biases were cataloged in RAG Part 11; we do not re-list them here.)")

## Demo 3 - The regression gate: replay, assert, report cost-per-success

`run_gate` is the CI gate. For a given token budget it grades every case, prints GREEN/RED with the reason a red case failed (trajectory, over token budget, component, or outcome), and reports two aggregate numbers: how many cases passed, and **cost-per-success** (total cost divided by greens). Cost-per-success (Part 11) is the number that tells you whether the suite is getting more expensive even when it still passes.

In [ ]:
def run_gate(token_budget):
    print(f"  operating envelope: <= {token_budget} tokens/run")
    results = [(c["name"], grade_case(c, token_budget)) for c in SUITE]
    for name, g in results:
        status = "GREEN" if g["pass"] else "RED"
        reason = ""
        if not g["pass"]:
            if not g["trajectory"]["pass"]:
                reason = "trajectory"
            elif not g["within_budget"]:
                reason = "over token budget"
            elif not g["component"]:
                reason = "component"
            else:
                reason = "outcome"
            reason = f" ({reason})"
        print(f"    [{status}] {name}  cost=${g['cost']:.5f}{reason}")
    greens = sum(1 for _n, g in results if g["pass"])
    total_cost = sum(g["cost"] for _n, g in results)
    cps = total_cost / greens if greens else float("inf")
    print(f"  gate: {greens}/{len(results)} green; cost-per-success ${cps:.5f}")
    return greens


print("run_gate ready (replay the suite; assert outcome + trajectory + envelope).")

## Demo 3 (continued) - Run the gate, then tighten the envelope

First the gate at **<= 400 tokens/run**: the good refund, the warranty lookup, and the security case are GREEN; the wrong-path refund is RED on **trajectory**. Three of four green, cost-per-success $0.00320.

Then we **tighten the envelope to <= 200 tokens/run**, simulating catching a cost regression. The good refund now goes RED, **over token budget**, even though its path is perfect: a three-step run at 80 tokens/step costs 240 tokens, above 200. The wrong-path run is still RED on trajectory. Two of four green, cost-per-success rises to $0.00480. That is the gate doing its job: a change that makes a green case pricier flips it red.

In [ ]:
print("\n" + bar)
print("3) THE REGRESSION GATE: replay the suite, assert outcome + trajectory + envelope.")
print(bar)
run_gate(token_budget=400)
print("\n  Now TIGHTEN the envelope (catch a cost regression):")
run_gate(token_budget=200)
print("  -> the wrong-path run was already red on trajectory; tightening tokens keeps the")
print("     gate honest about cost. A regression that makes a green case pricier flips it red.")

## Demo 4 - Verifiable success and security as eval

The security case is verifiable success made concrete. We grade the defended poisoned-ticket run and assert the **world state**: the ledger is empty (no unauthorized refund) and nothing was exfiltrated, so the case is GREEN. Success is a fact about the world (state + policy: the ledger holds exactly the authorized refunds, here none), not "the text said done." The caveats are real, a contaminated suite or a reward-hacked trajectory can pass the letter and miss the point, but the principle stands: a security guarantee you do not test is one you do not have.

In [ ]:
print("\n" + bar)
print("4) VERIFIABLE SUCCESS (state + policy) and SECURITY AS EVAL.")
print(bar)
sec = SUITE[3]
g = grade_case(sec, token_budget=400)
print(f"  security case '{sec['name']}': ledger={sec['state'] or '{}'} (no unauthorized refund), "
      f"no exfiltration -> {'GREEN' if g['pass'] else 'RED'}")
print("  Success is world STATE + POLICY (the ledger holds exactly the authorized refund),")
print("  not 'the text said done'. Caveats: a contaminated suite or a reward-hacked trajectory")
print("  can pass the letter and miss the point. A security guarantee you do not test you do not have.")

## The whole arc, Parts 1 to 17

One last print: the series, from a bare model and a loop to an agent that is robust, planful, reflective, remembering, bounded, durable, supervised, observable, protocol-speaking, sandboxed, multi-agent, secured, and now **graded**. Every part fixed one concrete failure of the part before it, by hand, offline, one mechanism at a time.

In [ ]:
print("\n" + bar)
print("THE WHOLE ARC, Parts 1 to 17")
print(bar)
print("  We started with a bare model and a loop (Part 1) and ended with an agent that is")
print("  robust, planful, reflective, remembering, bounded, durable, supervised, observable,")
print("  protocol-speaking, sandboxed, multi-agent, secured, and now GRADED. Every part fixed")
print("  one concrete failure of the part before it, by hand, offline, one mechanism at a time.")
print("  Eyeballing got us here; a three-layer eval + a golden-trajectory gate is what keeps it")
print("  here. Build it by hand, understand every line.")
print(bar)

## The finale: how you know it works

Sixteen parts gave the agent capabilities. This one gives you the only thing that lets you trust them: a way to **measure**. Eyeballing checks the answer and ships the wrong path. The fix is three orthogonal layers and a gate.

- **Three layers.** **OUTCOME** grades the world STATE, not the text (tau2-style). **TRAJECTORY** grades the PATH: set precision/recall, an order-aware edit distance, and the step budget, the layer that catches a correct refund reached the wrong way. **COMPONENT** unit-checks each call. A run can pass outcome and component and still **fail trajectory**, the run eyeballing would have shipped.
- **The judge is gameable for process.** A rubric LLM-judge rewards confident, verbose narration: the same bad path scored 0.6 terse and 0.9 verbose, while the programmatic guard held at FAIL. Use the judge for taste; use the guard for the path. (Answer-quality judge biases: RAG Part 11.)
- **The golden-trajectory gate.** A CI suite of `(task -> expected outcome + golden trajectory + operating envelope)`; replay, assert outcome + tool-call correctness + the envelope (steps, tool-calls, token-cost, timeout), and report **cost-per-success** (Part 11). Tightening the envelope flipped a perfectly-pathed run red on cost, that is how you catch a regression that quietly got more expensive.
- **Verifiable success + security as eval.** Success is STATE + POLICY, the ledger, not the transcript, with contamination and reward-hacking as the standing caveats. The Part 16 attack is just another case: assert no unauthorized refund and no exfiltration, because a security guarantee you do not test you do not have.
- **Your extension.** This graded single-shot trajectories. Multi-turn and long-horizon runs need trajectory eval over a whole conversation or a multi-day task, the same three layers, applied across turns.

**The seventeen-part arc closes here.** We started with a bare model in the smallest loop (Part 1) and, one concrete failure at a time, built robustness, planning, reflection, the four memories, compaction, budgets, durability, the human gate, observability, MCP, the code sandbox, supervisors, A2A, security, and now evaluation. Eyeballing got us through the build; a three-layer eval and a golden-trajectory gate are what keep it working after you stop watching. Build it by hand, understand every line.

**The full series:** [Agents from First Principles](https://www.mefby.com/essays/agents).